# Model to predict sleep state based on Actigraphy data

## Requirements

In [2]:
%pip install torch transformers pandas polars seaborn matplotlib pyarrow

  Using cached torch-2.7.1-cp313-cp313-win_amd64.whl.metadata (28 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached typing_extensions-4.14.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached PyYAML-6.0.2-cp313-cp313-win_amd64.whl.metadata (2.1 kB)
  Using cached regex-2024.11.6-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-win_amd64.whl

In [3]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\Usuario\OneDrive\Escritorio\UN\Distri\4-SEMESTRE\System-Analysis-N-Design\Systems-Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A. Data acquisition
We optain the parquet with the data with:
- **ENMO**: Euclidean Norm Minus One, a measure of physical activity.
- **Anglez**: Angle of the wrist, which can indicate the position of the wrist.
- **Step**: step identifier.
- **Timestamp**: time of the measurement.

In [ ]:
df = pl.read_parquet("data/train_series.parquet")
print(df.head())

shape: (5, 5)
┌──────────────┬──────┬──────────────────────────┬────────┬────────┐
│ series_id    ┆ step ┆ timestamp                ┆ anglez ┆ enmo   │
│ ---          ┆ ---  ┆ ---                      ┆ ---    ┆ ---    │
│ str          ┆ u32  ┆ str                      ┆ f32    ┆ f32    │
╞══════════════╪══════╪══════════════════════════╪════════╪════════╡
│ 038441c925bb ┆ 0    ┆ 2018-08-14T15:30:00-0400 ┆ 2.6367 ┆ 0.0217 │
│ 038441c925bb ┆ 1    ┆ 2018-08-14T15:30:05-0400 ┆ 2.6368 ┆ 0.0215 │
│ 038441c925bb ┆ 2    ┆ 2018-08-14T15:30:10-0400 ┆ 2.637  ┆ 0.0216 │
│ 038441c925bb ┆ 3    ┆ 2018-08-14T15:30:15-0400 ┆ 2.6368 ┆ 0.0213 │
│ 038441c925bb ┆ 4    ┆ 2018-08-14T15:30:20-0400 ┆ 2.6368 ┆ 0.0215 │
└──────────────┴──────┴──────────────────────────┴────────┴────────┘
